# Connecting to Estates data lakehouse

This notebooks walks through the installation and connection process for accessing data from a ducklake hosted via UoE [DataSync](https://information-services.ed.ac.uk/computing/desktop-personal/datasync) and an on-premises catalog database, using a Windows PC. It also provides some example queries.

You can check out some docs for the available data [here](http://129.215.207.51:8001/#!/overview).

> Note: you will need to have network access and an access key to the catalog database, and access granted to the storage layer, to use this guide.

## Installation

It's recommended to access the DataSync storage layer via [rclone](https://rclone.org/), due to limitations in the built-in Windows WebDAV client. This requires [installing rclone](https://rclone.org/install/) and [WinFsp](https://winfsp.dev/rel/).

You can skip this step and go straight to Mounting if you would prefer to use the Windows client.

#### rclone

Install rclone using [their instructions](https://rclone.org/install/), or using winget:

In [ ]:
!winget install Rclone.Rclone

#### WinFsp
Install WinFsp using the [instructions](https://winfsp.dev/rel/), or again using winget:

In [ ]:
!winget install -e --id WinFsp.WinFsp

## Mounting

### Using rclone

If you installed rclone, ensure you have a `.env` file like the following in the current directory with your credentials included. You can generate the value for `RCLONE_WEBDAV_PASS` by pasting your password in the next cell and running it:

In [ ]:
!rclone obscure "<your password here>"

Create a `.env` with the following structure:


Run the following cell to mount your DataSync directory to the `L:` drive, using rclone.

> Note: this cell will continue running for the duration of the mount. If you stop its execution, the drive will unmount. If you would like to run the mount in an external terminal instead, you can open a powershell window and run the `rclone_mount.ps1` file from this directory (or drag the `rclone_mount.ps1` file into the integrated terminal in vscode).

In [ ]:
import dotenv
import os

dotenv.load_dotenv()

print(f"Rclone WebDAV User: {os.getenv("RCLONE_WEBDAV_USER")}")

import subprocess

with subprocess.Popen(
    ['rclone', 'mount', ':webdav:', 'L:', '--vfs-cache-mode', 'writes', '--volname', 'DataSync'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
) as proc:
    for line in proc.stdout: # type: ignore
        print(line, end='', flush=True)

Rclone WebDAV User: estnet0
2026/04/23 13:02:19 NOTICE: Config file "C:\\Users\\ajohns3\\AppData\\Roaming\\rclone\\rclone.conf" not found - using defaults
2026/04/23 13:02:19 ERROR : symlinks not supported without the --links flag: /
The service rclone has been started.


### Using Windows WebDAV

Follow the instructions at https://information-services.ed.ac.uk/computing/desktop-personal/datasync/webdav to mount your DataSync directory as a local file. **Ensure and mount using the `L:` drive letter.**

### Verifying mount

If you navigate to `L:/` in the file explorer, you should see your DataSync files. This should include a `L:\datalake\EST` folder if you have access to the lake.

## Attaching to the lakehouse

### Catalog credentials

Add the following to the `.env` file in the current folder:

### DuckDB connection

Run the cell below to create a duckdb python connection to the lakehouse.

In [39]:
import duckdb
import os
import dotenv

dotenv.load_dotenv()

NETBIRD_HOST = os.getenv("NETBIRD_HOST")
CATALOG_DB_PORT = os.getenv("CATALOG_DB_PORT")
CATALOG_DB_USERNAME = os.getenv("CATALOG_DB_USERNAME")
DATA_PATH = 'L:/datalake/EST/'

print(f"Connecting to ducklake using catalog database at {NETBIRD_HOST}:{CATALOG_DB_PORT} with username {CATALOG_DB_USERNAME}")

con = duckdb.connect()
# note: to create persistent secrets that are available in other duckdb sessions on this machine, use PERSISTENT SECRET instead of SECRET
con.execute(f"""
    CREATE OR REPLACE SECRET ducklake_catalog 
            ( TYPE postgres, HOST '{NETBIRD_HOST}', PORT {CATALOG_DB_PORT}, 
            DBNAME 'Lakehouse_Catalog', USER '{CATALOG_DB_USERNAME}', PASSWORD '{os.getenv("CATALOG_DB_PASSWORD")}' );
""")
con.execute(f"""
 CREATE OR REPLACE SECRET 
        ( TYPE ducklake, DATA_PATH '{DATA_PATH}', METADATA_PATH '', 
            METADATA_PARAMETERS MAP {{'TYPE': 'postgres', 'SECRET': 'ducklake_catalog'}} );
""")
# note: if you change the above commands to use PERSISTENT SECRET, the below ATTACH command will work in any duckdb session on this machine.
con.execute("ATTACH 'ducklake:' AS uoe_est_lakehouse;")
con.execute("USE uoe_est_lakehouse;")

Connecting to ducklake using catalog database at 100.105.144.159:5433 with username read_only_user


## Examples

The below cells are example python scripts that demonstrate how to interact with the lakehouse.

In [28]:
schemas_query = con.execute("select distinct schema from (show all tables) where database != '__ducklake_metadata_uoe_est_lakehouse';")
schemas = schemas_query.df()
schemas.head(6)

,schema
0,raw_metabase
1,dbt_marts
2,dbt_int
3,dbt_staging
4,load
5,_dlt_staging_


In [36]:
# print all tables in dbt_marts
tables_query = con.execute("from (show all tables) where schema = 'dbt_marts';")

tables = tables_query.df()
tables.head(6)

,database,schema,name,column_names,column_types,temporary
0,uoe_est_lakehouse,dbt_marts,uoeap__utility_consumption,"[reading_datetime, reading_date, consumption, ...","[TIMESTAMP WITH TIME ZONE, TIMESTAMP WITH TIME...",False


In [41]:
# describe columns in dbt_marts.uoeap__utility_consumption
tables_query = con.execute("describe dbt_marts.uoeap__utility_consumption;")
tables = tables_query.df()
tables.head(20)

,column_name,column_type,null,key,default,extra
0,reading_datetime,TIMESTAMP WITH TIME ZONE,YES,None,NULL,None
1,reading_date,TIMESTAMP WITH TIME ZONE,YES,None,NULL,None
2,consumption,DOUBLE,YES,None,NULL,None
3,meter_reference,VARCHAR,YES,None,NULL,None
4,meter_channel_id,BIGINT,YES,None,NULL,None
5,channel_description,VARCHAR,YES,None,NULL,None
6,meter_type,VARCHAR,YES,None,NULL,None
7,meter_status,VARCHAR,YES,None,NULL,None
8,reading_status,VARCHAR,YES,None,NULL,None
9,account_units,VARCHAR,YES,None,NULL,None


In [63]:
# get meters in old college

old_college_meters_query = con.execute("""
select distinct meter_reference, channel_description, account_units, is_main_site_utility_account
from dbt_marts.uoeap__utility_consumption
where site_code = '0001';
""")
old_college_meters = old_college_meters_query.df()
old_college_meters.head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,meter_reference,channel_description,account_units,is_main_site_utility_account
0,0001NH001M,Heat Meter (kWh) RAW,kWh,False
1,0001NW002M,Water (Cum),Cu Metres,False
2,0001NE005S,Electricity (kWh),kWh,False
3,0001NE008S,Electricity (kWh),kWh,False
4,0001NH001M,Heat Meter (kWh),kWh,False
5,0001NE002M,Electricity (kWh),kWh,False
6,1800035330463,Electricity (kWh),kWh,False
7,0001NG001M,Natural Gas (CuFt),kWh,False
8,1800035330472,Electricity (kVArh Export),kWh,False
9,0001NE002M-Old,Electricity (kWh),kWh,False


In [ ]:
# energy by utility in old college for the last 30 days
energy_by_utility_query = con.execute("""
select utility, sum(consumption) as total_consumption
from dbt_marts.uoeap__utility_consumption
where site_code = '0001'
and reading_date >= current_date - interval '30' day
and is_main_site_utility_account = true
group by utility;
""")
energy_by_utility = energy_by_utility_query.df()
energy_by_utility.head(20)

BinderException: Binder Error: Referenced column "consumption_date" not found in FROM clause!
Candidate bindings: "consumption", "account_units", "site_code", "site_name", "channel_description"

LINE 5: and consumption_date >= current_date - interval '30' day
            ^

In [ ]:
# save all meter readings for old college to a dataframe

old_college_readings_query = con.execute("""
from dbt_marts.uoeap__utility_consumption
where site_code = '0001'
""")
old_college_readings = old_college_readings_query.df()
old_college_readings.head(20)

In [62]:
data_files_query = con.execute("""
select * from __ducklake_metadata_uoe_est_lakehouse.ducklake_data_file
                                       """)

data_files = data_files_query.df()
data_files.tail(10)

,data_file_id,table_id,begin_snapshot,end_snapshot,file_order,path,path_is_relative,file_format,record_count,file_size_bytes,footer_size,row_id_start,partition_id,encryption_key,mapping_id,partial_max
994,1291,117,1323,1343,<NA>,ducklake-019daea4-00ab-7f8e-9559-53a38b03ae37.parquet,True,parquet,538548,6656465,8125,1462043,<NA>,None,<NA>,<NA>
995,1292,117,1323,1343,<NA>,ducklake-019daea4-1f91-7e2e-92a4-e02ea9881943.parquet,True,parquet,561928,6750112,8125,2000591,<NA>,None,<NA>,<NA>
996,1293,117,1323,1343,<NA>,ducklake-019daea4-3ec4-7cd5-b371-dcc327f00c4b.parquet,True,parquet,555549,6842019,8123,2562519,<NA>,None,<NA>,<NA>
997,1314,136,1343,<NA>,<NA>,ducklake-019db055-8a28-7d60-bc73-7c6c22dbfefe.parquet,True,parquet,452365,5386187,6846,0,<NA>,None,<NA>,<NA>
998,1315,136,1343,<NA>,<NA>,ducklake-019db055-af63-78c3-b540-b89bb8932692.parquet,True,parquet,476889,5915820,6833,452365,<NA>,None,<NA>,<NA>
999,1316,136,1343,<NA>,<NA>,ducklake-019db055-dad4-7de2-a3f1-81eb9f0bde24.parquet,True,parquet,530452,6290394,8139,929254,<NA>,None,<NA>,<NA>
1000,1317,136,1343,<NA>,<NA>,ducklake-019db056-01dc-747e-ae75-2ef3d989934f.parquet,True,parquet,544252,6681383,8123,1459706,<NA>,None,<NA>,<NA>
1001,1318,136,1343,<NA>,<NA>,ducklake-019db056-2207-7af8-8924-f3f17fdddb82.parquet,True,parquet,561928,6750122,8123,2003958,<NA>,None,<NA>,<NA>
1002,1319,136,1343,<NA>,<NA>,ducklake-019db056-43eb-7f98-9f6d-502a78463f2c.parquet,True,parquet,557840,6870692,8126,2565886,<NA>,None,<NA>,<NA>
1003,1322,277,1608,<NA>,<NA>,ducklake-019dbad8-4212-7816-86a1-7d71b21ead6a.parquet,True,parquet,513838416,4178550946,6251236,0,<NA>,None,<NA>,<NA>
